# KaushalDub GPU worker (Kaggle / Colab)

Runs **the same worker code** as the Hugging Face Space (`backend/app/worker.py`), but on a GPU and listening on the
Redis queue **`gpu`**. Jobs land there when a trainer uses the *Free local models* plan or picks Indic Parler-TTS /
IndicConformer, and the API has `GPU_QUEUE_ENABLED=true`.

What you need:
1. Kaggle notebook with **GPU T4** and **Internet on**.
2. Kaggle *Add-ons → Secrets* (never paste keys into cells): `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`, `REDIS_URL`,
   `KEY_ENCRYPTION_SECRET` (same values as the Space), `HF_TOKEN` (for the gated AI4Bharat models). Optional:
   `SARVAM_API_KEY`, `GEMINI_API_KEY`, `GROQ_API_KEY` (platform keys for free-trial jobs).
3. The code: either set `REPO_URL` below to your git repo, or attach this project as a Kaggle dataset.
4. The step model bundle: attach `kaushaldub_model` (config.json, model.pt, kd_model.py, kd_metrics.py) as a dataset,
   or set the `STEP_MODEL_REPO` secret to a Hugging Face repo that holds it.

The worker keeps running until you stop the cell. Kaggle sessions end after ~12 h; just run it again.

In [ ]:
REPO_URL = ""                      # e.g. "https://github.com/<you>/kaushaldub.git" (leave empty to use a dataset)
CODE_DATASET = "/kaggle/input/kaushaldub-code"      # used when REPO_URL is empty
MODEL_DATASET = "/kaggle/input/kaushaldub-model"    # folder with config.json + model.pt (optional)
QUEUES = "gpu"                     # add ",cpu" to also take CPU jobs while this notebook runs
WORK = "/kaggle/working" if __import__("os").path.isdir("/kaggle/working") else "/content"

In [ ]:
# Secrets -> environment variables (values are never printed)
import os
names = ["SUPABASE_URL", "SUPABASE_SERVICE_KEY", "REDIS_URL", "KEY_ENCRYPTION_SECRET", "HF_TOKEN",
         "SARVAM_API_KEY", "GEMINI_API_KEY", "GROQ_API_KEY", "STEP_MODEL_REPO", "SUPABASE_JWT_SECRET"]
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    get = lambda n: sc.get_secret(n)
except ImportError:                                   # Colab
    from google.colab import userdata
    get = lambda n: userdata.get(n)
for n in names:
    try:
        v = get(n)
        if v:
            os.environ[n] = v
    except Exception:
        pass
missing = [n for n in ["SUPABASE_URL", "SUPABASE_SERVICE_KEY", "REDIS_URL", "KEY_ENCRYPTION_SECRET"] if not os.environ.get(n)]
print("secrets loaded:", sorted(n for n in names if os.environ.get(n)))
assert not missing, f"add these secrets first: {missing}"
os.environ.update({"WORKER_KIND": "gpu", "GPU_QUEUE_ENABLED": "true", "DEV_AUTH": "false", "MOCK_PROVIDERS": "false",
                   "STEP_MODEL": "bundle", "DATA_DIR": f"{WORK}/kd_data"})

In [ ]:
# Get the code (same repo as the Space)
import os, shutil, subprocess
APP = f"{WORK}/kaushaldub"
if not os.path.exists(APP):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, APP], check=True)
    else:
        shutil.copytree(CODE_DATASET, APP)
print(sorted(os.listdir(APP)))

In [ ]:
# Dependencies. Kaggle already has torch + CUDA; this adds the API/worker packages and the model libraries.
!pip install -q -r {APP}/backend/requirements.txt open_clip_torch accelerate sentencepiece onnxruntime virtualenv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Switch the notebook accelerator to GPU"

In [ ]:
# Step model bundle -> ml/kaushaldub_model (the worker also downloads it from STEP_MODEL_REPO if set)
import os, shutil
dst = f"{APP}/ml/kaushaldub_model"
if os.path.exists(f"{MODEL_DATASET}/model.pt") and not os.path.exists(f"{dst}/model.pt"):
    shutil.copytree(MODEL_DATASET, dst, dirs_exist_ok=True)
if not os.path.exists(f"{dst}/model.pt") and os.environ.get("STEP_MODEL_REPO"):
    from huggingface_hub import snapshot_download
    snapshot_download(os.environ["STEP_MODEL_REPO"], local_dir=dst, token=os.environ.get("HF_TOKEN"))
print("bundle files:", sorted(os.listdir(dst)) if os.path.exists(dst) else "MISSING")
os.environ["MODEL_BUNDLE_DIR"] = dst

In [ ]:
# Quick checks: Redis reachable, ffmpeg present, pipeline importable
import subprocess, sys
r = subprocess.run([sys.executable, "-c", (
    "from app.queue import redis_conn; print('redis ping:', redis_conn().ping());"
    "from app.mlpath import kp; print('ffmpeg:', kp.ffmpeg_exe());"
    "from app.queue import worker_status; print('workers online:', worker_status())")],
    cwd=f"{APP}/backend", capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])

In [ ]:
# Run the worker (blocks; stop the cell to stop the worker). Logs stream below; keys are never logged.
import subprocess, sys
p = subprocess.Popen([sys.executable, "-u", "-m", "app.worker", "--queues", QUEUES, "--kind", "gpu"],
                     cwd=f"{APP}/backend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in p.stdout:
        print(line, end="")
except KeyboardInterrupt:
    p.terminate()
    print("worker stopped")